# Notebook 06: Clinical Robustness Stress Testing & Domain Shift
### Evaluating High-Frequency Noise, Baseline Wander, Missing Leads, and Held-Out Acquisition Sites

---

### Mathematical Formulations of Stress Corruptions
All perturbation parameters were pre-specified based on clinical ECG artifact literature (Strodthoff et al., 2021) prior to test cohort evaluation:
1. **High-Frequency EMG / Tremor Noise**:
   $$X_{\text{EMG}} = X + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2), \quad \sigma = 0.15$$
2. **Low-Frequency Baseline Wander**:
   $$X_{\text{drift}}[c, t] = X[c, t] + A \sin\left(2\pi f \frac{t}{f_s} + \phi_c\right)$$
   where $A = 0.50$, $f = 0.20$ Hz, $f_s = 100$ Hz, and $\phi_c \sim \mathcal{U}(0, 2\pi)$ per lead.
3. **Missing Precordial Leads (V1–V3)**:
   $$X_{\text{precordial}}[c, t] = 0 \quad \forall c \in \{6, 7, 8\}, \quad t \in [0, 1000)$$
4. **Missing Limb Leads (I–III)**:
   $$X_{\text{limb}}[c, t] = 0 \quad \forall c \in \{0, 1, 2\}, \quad t \in [0, 1000)$$
5. **Acquisition Site Domain Shift**:
   Held-out test records acquired from non-primary clinical recording sites (`site != 0.0`) within PTB-XL.


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('.'))

import pandas as pd
from IPython.display import Image, display

print("Modules loaded.")


## 1. Robustness Benchmark Table Across 5 Seeds


In [ ]:
robustness_summary = pd.read_csv('results/tables/robustness_multiseed_summary.csv')
print("Diagnostic Retention Under Clinical Degradations Across 5 Seeds (Mean ± SD):")
robustness_summary[['Condition', 'Single Large AUROC', 'MOSAIC-Med AUROC', 'Delta AUROC (M - S)', 'Holm p-value', 'Significant (α=0.05)']]


## 2. Retention Comparison Under Stress Corruptions


In [ ]:
display(Image('results/images/robustness_stress.png'))


## 3. Robustness Findings & Scope Calibration
- **Resilience to Sensor Loss**: Under precordial lead loss (V1–V3), Single Large drops to $0.8694 \pm 0.0021$, whereas MOSAIC-Med retains $0.8801 \pm 0.0030$ ($\Delta = +0.0106$, Holm $p = 8.82 \times 10^{-4}$). Ensembling orthogonal inductive biases provides compensatory feature extraction across unaffected leads.
- **Acquisition Site Shift**: Under held-out non-primary site records (`site != 0`), MOSAIC-Med achieves AUROC $0.9033 \pm 0.0009$ vs $0.8978 \pm 0.0008$ ($p = 3.72 \times 10^{-5}$).
- **Validation Scope**: This held-out site evaluation tests internal acquisition transferability within PTB-XL; prospective validation on independent hospital cohorts remains required before clinical claims can be made.
